# Strava OAuth (alternative to tools/strava_auth.py)

The recommended way to connect or reconnect Strava is the local command `python tools/strava_auth.py`, which opens
Strava's consent page and stores the credentials in Key Vault automatically. Use this notebook only if you cannot run
Python locally.

Exchanges a one-time Strava authorization code for a refresh token and stores it in **Azure Key Vault**
(`key_vault_url`). Nothing sensitive is written to the lakehouse or printed.

**First-time setup** (once): store your Strava API app credentials in the vault:

```powershell
az keyvault secret set --vault-name <your-vault> --name strava-client-id --value <CLIENT_ID>
az keyvault secret set --vault-name <your-vault> --name strava-client-secret --value <CLIENT_SECRET>
```

**Authorize / re-authorize**

1. Open (replace `<CLIENT_ID>`):
   `https://www.strava.com/oauth/authorize?client_id=<CLIENT_ID>&response_type=code&redirect_uri=http://localhost/exchange_token&approval_prompt=force&scope=read,read_all,profile:read_all,activity:read_all`
2. Click **Authorize** and copy the `code=` value from the redirect URL.
3. Run this notebook with the `authorization_code` parameter set.

In [ ]:
authorization_code = ""
key_vault_url = "https://your-key-vault.vault.azure.net/"

In [ ]:
import json
import requests

# Strava credentials live only in Azure Key Vault - never in lakehouse tables, notebook outputs or source control.
# Secrets: strava-client-id, strava-client-secret, strava-refresh-token. The running identity needs
# "Key Vault Secrets Officer" on the vault (read + write, because Strava rotates refresh tokens).
def kv_get(name):
    return notebookutils.credentials.getSecret(key_vault_url, name)


def kv_set(name, value):
    kv_token = notebookutils.credentials.getToken("keyvault")
    r = requests.put(
        f"{key_vault_url.rstrip('/')}/secrets/{name}?api-version=7.4",
        headers={"Authorization": f"Bearer {kv_token}"},
        json={"value": value, "contentType": "text/plain"},
        timeout=30,
    )
    if r.status_code >= 300:
        raise RuntimeError(f"Key Vault write failed for {name}: HTTP {r.status_code}")


if not authorization_code:
    result = {"status": "skipped", "requires_authorization": True,
              "message": "Provide authorization_code to store a new Strava refresh token in Key Vault."}
else:
    resp = requests.post("https://www.strava.com/oauth/token", data={
        "client_id": kv_get("strava-client-id"),
        "client_secret": kv_get("strava-client-secret"),
        "code": authorization_code,
        "grant_type": "authorization_code",
    }, timeout=60)
    if resp.status_code != 200:
        raise RuntimeError(f"Strava token exchange failed: HTTP {resp.status_code}")
    token_data = resp.json()
    kv_set("strava-refresh-token", token_data["refresh_token"])
    athlete = token_data.get("athlete") or {}
    result = {"status": "refresh_token_saved_to_key_vault", "athlete_id": athlete.get("id")}

print(json.dumps(result, indent=2))
notebookutils.notebook.exit(json.dumps(result))